# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedAmmarSharif/Flyrank-ML-Internship-Work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. My Lane as an ML Task

**Selected Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**ML Task Type:** Classification used for ranking.

My goal is to classify each content page as either declining or not declining based on its available search performance, engagement, and content-related signals.

The model's predicted probability of decline can then be used as a priority score. Pages with higher probabilities can be ranked first for human review.

The final output will be a ranked list of content pages that helps content strategists decide which pages should be investigated first.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Target or Proxy

**Target column:** `is_declining_label`

The target will be created from the existing `trend_direction` column:

- `1` means the page is declining (`trend_direction == "down"`).
- `0` means the page is not declining.

This is a proxy for identifying pages that may need attention. It does not mean every declining page needs an immediate refresh.

The model will learn patterns from available features such as impressions, clicks, average search position, CTR, engagement rate, and content age.

**Important:** `trend_direction` and `trend_pct` will not be used as input features because they are directly related to the target and could cause data leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 3. Success Metric

**Primary metric: Precision@50**

Precision@50 measures how many of the 50 highest-priority pages identified by the model are actually declining.

For example, a Precision@50 of 0.70 means that 35 out of the 50 selected pages are declining.

This metric is useful because a content team has limited time and cannot investigate every page at once.

I will compare the model against a simple baseline and evaluate it using a client-level holdout split so that pages from the same client do not appear in both training and testing data.

Accuracy alone is not sufficient because the main goal is to produce a useful priority list.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/SyedAmmarSharif/Flyrank-ML-Internship-Work/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Total content pages:", len(df))

df.head()

Dataset shape: (30000, 44)
Total content pages: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


An important limitation is that the target represents an existing decline label in the dataset, not a guaranteed prediction of future decline. The model will be evaluated on unseen client data, and its performance will be compared with a fixed-rule baseline. Machine Learning will only be considered useful if it demonstrates a meaningful improvement in the evaluation.

### 5. Why ML Beats a Fixed Rule Here

A fixed rule can prioritize pages using one or two conditions, such as pages with low impressions or pages that have not been updated recently.

However, content performance can depend on several signals together. A page may have high impressions but low CTR, or it may have good engagement despite being old.

Machine Learning can learn patterns from multiple features and identify combinations that a simple fixed rule may miss.

The model is not automatically better than every rule. I will compare its performance against a baseline to determine whether it provides a useful improvement.

The output will support human decision-making. A content strategist can investigate high-priority pages and decide whether to refresh the content, improve its search presentation, monitor it, or leave it unchanged.



In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

selected_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "is_declining_label"
]

task_df = df[selected_columns].copy()

print("Unit of analysis: One row represents one content page.")
print("Task dataframe shape:", task_df.shape)

task_df.head(10)

Unit of analysis: One row represents one content page.
Task dataframe shape: (30000, 10)


,content_id,impressions_90d,clicks_90d,avg_position,ctr,engagement_rate,content_age_days,days_since_last_update,word_count,is_declining_label
0,content_304f48230142,3803,29,10.6,0.76,5.88,187,20,3221.0,1
1,content_a1fb4e703a9e,15320,7,20.3,0.05,0.00,445,25,2481.0,1
2,content_9aa793d4d895,12581,11,36.5,0.09,0.00,141,20,3515.0,1
3,content_331d6c4de07b,11751,58,6.2,0.49,1.28,463,22,NaN,0
4,content_d99b7a2d90ca,19140,24,44.0,0.13,0.00,263,14,2803.0,1
5,content_d4084a4bc775,3970,1,8.5,0.03,0.00,147,20,3080.0,1
6,content_9a34b442b552,20,0,7.0,0.00,0.00,90,20,3059.0,1
7,content_a63219c6e95a,1724,1,21.2,0.06,3.57,445,22,NaN,0
8,content_5e6c160719bc,32574,29,46.0,0.09,5.88,90,20,3807.0,1
9,content_c27558df2b0c,1240,2,4.9,0.16,0.00,257,104,NaN,1


In [9]:
print("Target distribution:")
print(task_df["is_declining_label"].value_counts())

print("\nTarget percentages:")
print(task_df["is_declining_label"].value_counts(normalize=True).mul(100).round(2))

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Final Verdict: Completed!